# Dataverse Explorer

This notebook demonstrates how to connect to Microsoft Dataverse and explore tables and schemas using the DataverseClient.

## Setup

Import the DataverseClient and initialize connection.

In [ ]:
import sys
sys.path.append('../src/')

from dataverse_client import DataverseClient
import pandas as pd

# Initialize client
client = DataverseClient()

# Test connection
if client.test_connection():
    print("✅ Successfully connected to Dataverse!")
else:
    print("❌ Connection failed - check your .env file")

## List All Tables

Get a list of all available tables in the Dataverse environment.

In [ ]:
# Get all tables
tables = client.list_tables()

print(f"Found {len(tables)} tables in Dataverse:\n")

# Create a clean list for display
table_list = []
for table in tables:
    logical_name = table.get('LogicalName', 'N/A')
    display_name = table.get('DisplayName', {}).get('UserLocalizedLabel', {}).get('Label', 'N/A')
    schema_name = table.get('SchemaName', 'N/A')
    entity_set = table.get('EntitySetName', 'N/A')
    
    table_list.append({
        'Logical Name': logical_name,
        'Display Name': display_name,
        'Schema Name': schema_name,
        'Entity Set': entity_set
    })

# Display as DataFrame for better formatting
df_tables = pd.DataFrame(table_list)
print(df_tables.to_string(index=False))

# Show first 10 for quick reference
print(f"\n\nFirst 10 tables:")
for i, table in enumerate(tables[:10]):
    logical_name = table.get('LogicalName')
    display_name = table.get('DisplayName', {}).get('UserLocalizedLabel', {}).get('Label', 'N/A')
    print(f"{i+1:2d}. {logical_name} - {display_name}")

## Table Schema Exploration

Get detailed schema information for a specific table. Change the `table_name` variable to explore different tables.

In [ ]:
# Choose a table to explore (change this to any table from the list above)
table_name = "account"  # Common table - change to explore others

print(f"📋 Getting schema for table: {table_name}\n")

try:
    # Get table schema
    schema = client.get_table_schema(table_name)
    
    # Display basic table information
    print("=== TABLE INFORMATION ===")
    print(f"Logical Name: {schema.get('LogicalName')}")
    print(f"Display Name: {schema.get('DisplayName', {}).get('UserLocalizedLabel', {}).get('Label', 'N/A')}")
    print(f"Schema Name: {schema.get('SchemaName')}")
    print(f"Entity Set Name: {schema.get('EntitySetName')}")
    print(f"Primary ID Field: {schema.get('PrimaryIdAttribute')}")
    print(f"Primary Name Field: {schema.get('PrimaryNameAttribute')}")
    
    # Get and display attributes
    attributes = schema.get('Attributes', [])
    print(f"\n=== ATTRIBUTES ({len(attributes)} total) ===")
    
    # Create attributes DataFrame
    attr_list = []
    for attr in attributes:
        logical_name = attr.get('LogicalName', 'N/A')
        display_name = attr.get('DisplayName', {}).get('UserLocalizedLabel', {}).get('Label', 'N/A')
        attr_type = attr.get('AttributeType', 'N/A')
        is_primary = "🔑" if attr.get('IsPrimaryId', False) else ""
        
        attr_list.append({
            'Field': logical_name,
            'Display Name': display_name,
            'Type': attr_type,
            'Primary': is_primary
        })
    
    df_attributes = pd.DataFrame(attr_list)
    print(df_attributes.to_string(index=False))
    
    # Show first 10 attributes in detail
    print(f"\n=== FIRST 10 ATTRIBUTES (DETAILED) ===")
    for i, attr in enumerate(attributes[:10]):
        logical_name = attr.get('LogicalName')
        display_name = attr.get('DisplayName', {}).get('UserLocalizedLabel', {}).get('Label', 'N/A')
        attr_type = attr.get('AttributeType')
        is_primary = " (PRIMARY KEY)" if attr.get('IsPrimaryId', False) else ""
        
        print(f"{i+1:2d}. {logical_name}: {attr_type} - {display_name}{is_primary}")
        
except Exception as e:
    print(f"❌ Error retrieving schema for '{table_name}': {e}")
    print("💡 Make sure the table name is correct and accessible")

## Quick Table Attributes

Get just the attributes for a table using the simpler method.

In [ ]:
# Get detailed attributes for the same table
print(f"🔍 Getting detailed attributes for: {table_name}\n")

try:
    attributes = client.get_table_attributes(table_name)
    
    print(f"Found {len(attributes)} attributes:\n")
    
    # Show first 15 with additional details
    for i, attr in enumerate(attributes[:15]):
        logical_name = attr.get('LogicalName')
        display_name = attr.get('DisplayName', {}).get('UserLocalizedLabel', {}).get('Label', 'N/A')
        attr_type = attr.get('AttributeType')
        schema_name = attr.get('SchemaName')
        can_create = "✓" if attr.get('IsValidForCreate', False) else "✗"
        can_update = "✓" if attr.get('IsValidForUpdate', False) else "✗"
        is_primary = "🔑" if attr.get('IsPrimaryId', False) else ""
        
        print(f"{i+1:2d}. {logical_name} ({schema_name})")
        print(f"    Type: {attr_type} | Create: {can_create} | Update: {can_update} {is_primary}")
        print(f"    Display: {display_name}")
        print()
        
except Exception as e:
    print(f"❌ Error: {e}")